<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "604"
iso_country = "Peru"
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2024
gdp = 295.200e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Peru': 

Number of trade records present in dataset: 43509
Number of partner countries present in dataset: 171
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$46380.7m
Total weight within the dataset: 43376.8 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,604,Peru,842,USA,Import,Sea,20120,020120 | Meat; of bovine anim,0201,0201 | Meat of bovine anima,02,02 | Meat and edible meat,21111,"Meat of bovine animals, fresh or chilled",Agriculture,19,CMT - Bovine meat prods,944493.401,916762.260,944493.401,kg,59046.141,59046.141,141768.459490,4732.0,2940.327572,4.168457e+05,['Container'],['Container'],1096.987620,1093.506439,1093.841765,1093.841765,1095.699090,1095.699090,1154.929971,2168.295216,1260.962819,1172.926743,1177.829825,1214.442525,1127.675937,2102.189661,1336.717794,1224.153508,1192.455074,1256.485922,1103.759734,2348.627253,1648.367709,1463.895608,1347.131810,1315.135603,1080.260278,2451.578544,1971.006122,1823.854596,1443.954468,1419.096507,1057.214553,2538.393371,2455.428807,2426.991812,1563.780957,1495.925716,1033.718612,2493.567539,2404.584980,2595.600486,1991.255237,1523.844332,1010.826701,2174.699574,2338.239472,2587.402902,1940.722515,1542.205253,988.311542,1930.217113,1973.456781,2363.975969,1892.677831,1553.915496,21111
1,1,604,Peru,032,Argentina,Import,Sea,20130,020130 | Meat; of bovine anim,0201,0201 | Meat of bovine anima,02,02 | Meat and edible meat,21111,"Meat of bovine animals, fresh or chilled",Agriculture,19,CMT - Bovine meat prods,2387965.625,2330624.877,2387965.625,kg,176518.620,176518.620,358433.640313,7748.0,4814.382508,1.725637e+06,['Container'],['Container'],4541.253499,4526.842280,4528.230447,4528.230447,4535.919305,4535.919305,4781.120292,8976.198137,5220.069676,4855.622412,4875.919943,5027.487333,4668.295432,8702.537729,5533.676267,5067.688365,4936.464805,5201.536447,4569.288352,9722.727524,6823.828717,6060.160507,5576.787677,5444.331410,4472.006499,10148.920035,8159.470793,7550.300397,5977.609189,5874.703470,4376.602979,10508.311640,10164.859159,10047.137137,6473.660787,6192.757118,4279.335679,10322.743943,9954.378473,10745.134741,8243.297048,6308.333180,4184.568913,9002.710573,9679.724719,10711.198800,8034.104257,6384.342786,4091.361802,7990.614529,8169.615907,9786.267357,7835.211323,6432.820254,21111


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $6020.5m (13.0%), ... 

                               cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                               
10 | Cereals               1.842822e+09  6.980286e+09      4.0     16.1
23 | Food industries, res  7.119354e+08  1.286976e+09      1.5      3.0
15 | Animal, vegetable or  6.372011e+08  6.099800e+08      1.4      1.4
21 | Miscellaneous edible  4.012471e+08  8.607525e+07      0.9      0.2
04 | Dairy produce; birds  3.141203e+08  1.092911e+08      0.7      0.3
22 | Beverages, spirits a  3.041978e+08  3.130473e+08      0.7      0.7
02 | Meat and edible meat  2.754820e+08  1.348651e+08      0.6      0.3
17 | Sugars and sugar con  2.703444e+08  4.384467e+08      0.6      1.0
19 | Preparations of cere  1.625862e+08  5.494469e+07      0.4      0.1
03 | Fish and crustaceans  1.347456e+08  3.240143e+07      0.3      0.1
20 | Preparations of vege  1.31824

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $4977.8m (10.7%), ... 

                      cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                     
Agriculture       2.825163e+09  5.831691e+09      6.1     13.4
Food & Beverages  2.126991e+09  2.156778e+09      4.6      5.0
Fishing           2.567044e+07  1.190287e+07      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $4704.2m (10.1%), ... 

                                   cifvalue    vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                  
OFD - Food products n.e.c.     1.348426e+09  6.192542e+08      2.9      1.4
VOL - Vegetable oils and fats  1.076435e+09  1.676513e+09      2.3      3.9
GRO - Cereal grains n.e.c.     9.970280e+08  4.121254e+09      2.1      9.5
MIL - Dairy products           3.191170e+08  1.119890e+08      0.7      0.3
OMT - Meat products n.e.c.     2.424358e+08  1.354518e+08      0.5      0.3
V_F - Vegetables, fruit, nuts  2.067194e+08  2.391131e+08      0.4      0.6
PCR - Processed rice           1.225801e+08  2.129217e+08      0.3      0.5
SGR - Sugar                    1.200169e+08  2.632527e+08      0.3      0.6
PFB - Plant-based fibers       9.250732e+07  4.755801e+07      0.2      0.1
CMT - Bovine meat prods        7.355445e+07  2.758477e+07      0.2      0.1
OCR - Crops n.e.c.   

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 13.0% 
GTAP 10.1% 
EORA 10.7%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_16224/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,604,Peru,842,USA,Import,Sea,20120,020120 | Meat; of bovine anim,0201,0201 | Meat of bovine anima,02,02 | Meat and edible meat,21111,"Meat of bovine animals, fresh or chilled",Agriculture,19,CMT - Bovine meat prods,944493.401,916762.260,944493.401,kg,59046.141,59046.141,141768.459490,4732.0,2940.327572,4.168457e+05,['Container'],['Container'],1096.987620,1093.506439,1093.841765,1093.841765,1095.699090,1095.699090,1154.929971,2168.295216,1260.962819,1172.926743,1177.829825,1214.442525,1127.675937,2102.189661,1336.717794,1224.153508,1192.455074,1256.485922,1103.759734,2348.627253,1648.367709,1463.895608,1347.131810,1315.135603,1080.260278,2451.578544,1971.006122,1823.854596,1443.954468,1419.096507,1057.214553,2538.393371,2455.428807,2426.991812,1563.780957,1495.925716,1033.718612,2493.567539,2404.584980,2595.600486,1991.255237,1523.844332,1010.826701,2174.699574,2338.239472,2587.402902,1940.722515,1542.205253,988.311542,1930.217113,1973.456781,2363.975969,1892.677831,1553.915496,21111,Meat and Prep
1,1,604,Peru,032,Argentina,Import,Sea,20130,020130 | Meat; of bovine anim,0201,0201 | Meat of bovine anima,02,02 | Meat and edible meat,21111,"Meat of bovine animals, fresh or chilled",Agriculture,19,CMT - Bovine meat prods,2387965.625,2330624.877,2387965.625,kg,176518.620,176518.620,358433.640313,7748.0,4814.382508,1.725637e+06,['Container'],['Container'],4541.253499,4526.842280,4528.230447,4528.230447,4535.919305,4535.919305,4781.120292,8976.198137,5220.069676,4855.622412,4875.919943,5027.487333,4668.295432,8702.537729,5533.676267,5067.688365,4936.464805,5201.536447,4569.288352,9722.727524,6823.828717,6060.160507,5576.787677,5444.331410,4472.006499,10148.920035,8159.470793,7550.300397,5977.609189,5874.703470,4376.602979,10508.311640,10164.859159,10047.137137,6473.660787,6192.757118,4279.335679,10322.743943,9954.378473,10745.134741,8243.297048,6308.333180,4184.568913,9002.710573,9679.724719,10711.198800,8034.104257,6384.342786,4091.361802,7990.614529,8169.615907,9786.267357,7835.211323,6432.820254,21111,Meat and Prep


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $4093.4m of $46380.7m (8.8%)
Volume: 8792.3 of 43376.8 million kilograms (20.3%)
Value-to-Weight: 0.47 from 1.07 US$ per kg (+-56.5%)
Average distance to market (volume-weighted): 4660.9 from 5175.7 nm (+-9.9%)

            food_group  N_com      cifvalue    vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     20  1.720242e+09  6.767364e+09      3.7     15.6
1    Processed Cereals    597  6.970951e+08  5.016841e+08      1.5      1.2
2        Fats and Oils    188  5.878047e+08  5.535054e+08      1.3      1.3
3                Dairy    110  3.128294e+08  1.088370e+08      0.7      0.3
4      Sugar and Honey    115  2.708847e+08  4.386016e+08      0.6      1.0
5        Meat and Prep     40  2.483694e+08  1.492242e+08      0.5      0.3
6           Vegetables    160  1.510470e+08  1.957168e+08      0.3      0.5
7               Fruits     80  1.051170e+08  7.740086e+07      0.2      0.2



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.445110e+08,2.653052e+08,1.569812e+08,1.463846e+08,1.469573e+08,1.515881e+08,1.328815e+08,3.093126e+08,2.987144e+08,2.968922e+08,1.933318e+08,1.852814e+08,1.246297e+08,2.379098e+08,2.435118e+08,2.903339e+08,2.333564e+08,1.924366e+08
1,Fats and Oils,1.277327e+09,2.051586e+09,1.356278e+09,1.274035e+09,1.277584e+09,1.324400e+09,1.201541e+09,2.313043e+09,2.217863e+09,2.287523e+09,1.624744e+09,1.541552e+09,1.148576e+09,1.918349e+09,1.980074e+09,2.301924e+09,1.902258e+09,1.587183e+09
2,Fruits,7.376386e+07,1.384107e+08,8.052558e+07,7.490828e+07,7.522127e+07,7.756037e+07,6.753002e+07,1.620388e+08,1.567303e+08,1.549402e+08,9.985606e+07,9.551763e+07,6.313426e+07,1.232437e+08,1.260070e+08,1.509183e+08,1.208458e+08,9.921934e+07
3,Meat and Prep,1.797750e+08,3.375141e+08,1.962799e+08,1.825763e+08,1.833395e+08,1.890386e+08,1.645647e+08,3.951232e+08,3.822090e+08,3.777825e+08,2.434162e+08,2.328539e+08,1.538394e+08,3.004552e+08,3.071858e+08,3.679736e+08,2.946119e+08,2.418806e+08
4,Processed Cereals,7.739454e+08,1.436262e+09,8.426797e+08,7.848788e+08,7.881276e+08,8.128323e+08,7.100432e+08,1.682260e+09,1.624641e+09,1.611337e+09,1.043565e+09,9.968432e+08,6.649795e+08,1.285381e+09,1.314701e+09,1.569543e+09,1.260216e+09,1.035205e+09
5,Sugar and Honey,3.170245e+08,5.493259e+08,3.398811e+08,3.188590e+08,3.200712e+08,3.307364e+08,2.944733e+08,6.486624e+08,6.237678e+08,6.292264e+08,4.233143e+08,3.949371e+08,2.786188e+08,5.105343e+08,5.224586e+08,6.094902e+08,4.997290e+08,4.079800e+08
6,Unprocessed Cereals,8.694666e+09,1.562478e+10,9.397787e+09,8.782835e+09,8.817659e+09,9.102826e+09,8.024087e+09,1.838073e+10,1.773110e+10,1.771515e+10,1.169333e+10,1.102184e+10,7.552058e+09,1.424306e+10,1.456679e+10,1.719861e+10,1.395057e+10,1.141189e+10
7,Vegetables,2.022908e+08,3.774201e+08,2.205354e+08,2.052845e+08,2.061381e+08,2.125740e+08,1.853989e+08,4.419469e+08,4.271134e+08,4.229558e+08,2.732608e+08,2.612277e+08,1.734863e+08,3.369290e+08,3.445550e+08,4.119943e+08,3.303547e+08,2.713205e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.39,0.04,0.01,0.01,0.02,0.56,0.53,0.52,0.19,0.17,0.36,0.38,0.53,0.35,0.22
1,Fats and Oils,1.32,0.13,-0.01,0.00,0.08,1.89,1.73,1.85,0.72,0.58,1.31,1.41,1.96,1.28,0.75
2,Fruits,0.61,0.06,0.01,0.01,0.04,0.90,0.85,0.83,0.31,0.27,0.57,0.60,0.84,0.55,0.34
3,Meat and Prep,0.64,0.07,0.01,0.01,0.04,0.93,0.88,0.86,0.32,0.27,0.59,0.62,0.86,0.57,0.35
4,Processed Cereals,0.95,0.10,0.02,0.02,0.06,1.39,1.31,1.29,0.48,0.41,0.89,0.93,1.30,0.85,0.53
5,Sugar and Honey,0.86,0.08,0.01,0.01,0.05,1.31,1.22,1.24,0.48,0.37,0.86,0.90,1.22,0.82,0.48
6,Unprocessed Cereals,4.03,0.41,0.05,0.07,0.24,6.02,5.64,5.63,2.13,1.74,3.89,4.08,5.61,3.72,2.24
7,Vegetables,1.16,0.12,0.02,0.03,0.07,1.70,1.60,1.57,0.58,0.50,1.08,1.13,1.58,1.04,0.65
